In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

from model import DR_EfficientNet_CBAM
from data_loader import get_dataloaders


In [2]:
device = torch.device("cpu")
print("Using device:", device)


Using device: cpu


In [3]:
DATASET_PATH = "Datasets/Final_DR_Dataset"

train_loader, val_loader, test_loader = get_dataloaders(
    DATASET_PATH,
    batch_size=8
)

print("Dataloaders loaded")


Dataloaders loaded


In [4]:
model = DR_EfficientNet_CBAM(num_classes=5).to(device)
print("Model created")


Model created


c:\Users\dhira_5fqr2uc\Desktop\Diabetic _Retinography_Project\dr_env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dhira_5fqr2uc\Desktop\Diabetic _Retinography_Project\dr_env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
for param in model.features.parameters():
    param.requires_grad = False

print("EfficientNet backbone frozen")


EfficientNet backbone frozen


In [6]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


In [7]:
def train_one_epoch(model, loader):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    return running_loss / len(loader), acc


In [8]:
def validate(model, loader):
    model.eval()
    running_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    return running_loss / len(loader), acc
